In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import xarray as xr
import dask.array as da
from polsarpro.util import boxcar, C3_to_T3, S_to_C3, S_to_T3, C4_to_T4, T3_to_C3
from polsarpro.io import open_netcdf_beam
from polsarpro.classification import wishart_h_a_alpha
# from polsarpro.classification import _reconstruct_matrix_from_ds
from pathlib import Path

# optional import for progress bar
from dask.diagnostics import ProgressBar


# change to your data paths
# original dataset
input_alos_data = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

# S2 inputs generated with C-PSP
input_test_dir = Path("/data/psp/SAN_FRANCISCO_ALOS1/")

# # output files from C
# output_test_dir = Path("/data/psp/res/vanzyl_c/")
# if not os.path.isdir(output_test_dir):
#     os.mkdir(output_test_dir)

In [ ]:
# uncomment to test on S matrix made with SNAP
S = open_netcdf_beam(input_alos_data)
label = wishart_h_a_alpha(S)
# C3 = boxcar(S_to_C3(S), 5, 5)

In [ ]:
with ProgressBar():
    label = label.compute()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.imshow(label.label, interpolation="none")
plt.colorbar(fraction=0.046, pad=0.04)

In [ ]:
shp = C3.m11.shape
# to be replaced by h_a_alpha initial classes
class_map = da.random.randint(1, 9, size=shp)#.chunks(C3.chunks)

In [ ]:
def _trace_product_3_wish(M_inv, cov_ds):

    m11 = cov_ds.m11.data
    m12 = cov_ds.m12.data
    m22 = cov_ds.m22.data
    m13 = cov_ds.m13.data
    m23 = cov_ds.m23.data
    m33 = cov_ds.m33.data
    
    trace = (M_inv[0, 0] * m11 +
             M_inv[0, 1] * da.conj(m12) +
             M_inv[0, 2] * da.conj(m13) +
             M_inv[1, 0] * m12 +
             M_inv[1, 1] * m22 +
             M_inv[1, 2] * da.conj(m23) +
             M_inv[2, 0] * m13 +
             M_inv[2, 1] * m23 +
             M_inv[2, 2] * m33)
    
    return trace

In [ ]:
nclass = 8
centers = []
inv_centers = []
dist_min = da.full_like(C3.m11, fill_value=da.inf)
label_min = class_map.copy()
for i in range(1, nclass + 1):
    # class center
    mask = class_map == i
    npts = mask.sum()
    # center = xr.where(class_map == i, C3.chunk(dict(x=1024,y=1024)), 0).sum() / npts
    center = xr.where(class_map == i, C3, 0).sum() / npts
    # TODO: hardcode chunk shape (C3 T3 -> 3x3, C4 T4 -> 4x4)
    M = _reconstruct_matrix_from_ds(center).rechunk("auto") + 1e-12*da.eye(3)

    # compute distance components
    meta = (np.array([], dtype="complex64").reshape((0, 0)),)
    M_inv = da.apply_gufunc(np.linalg.inv, "(i,j)->(i,j)", M, meta=meta)
    M_det = da.apply_gufunc(np.linalg.det, "(i,j)->()", M, meta=meta)

    # update classification 
    dist = da.log(M_det.real) + _trace_product_3_wish(M_inv, C3)
    is_smaller = dist < dist_min
    label_min = da.where(is_smaller, i, label_min)
    dist_min = da.where(is_smaller, dist, dist_min)



In [ ]:
with ProgressBar():
    # center = center.compute()
    # centers = da.compute(centers)
    label_map = label_min.compute()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.imshow(label_map[::8], interpolation="none")
plt.colorbar(fraction=0.046, pad=0.04)